In [ ]:
# Cell 1: Change to the correct directory
import os
from repo_paths import bootstrap, REPO_ROOT, FROZEN_INST
bootstrap()
print("Current directory:", os.getcwd())

# Cell 2: Add the directory to Python path (if needed)
import sys
sys.path.insert(0, '.')  # Add current directory
sys.path.append('jax_ib/')  # Add jax_ib path
sys.path.insert(0, 'jax-cfd')  # Add jax-cfd path
print("Python path updated")



from plot_tbnn_training_results import plot_individual_flow_fields
print("✅ Successfully imported plotting!")

from porous_media_flow import *
print("✅ Successfully imported porous media flow demo!")

#from serpentine_flow import *
#print("✅ Successfully imported serpentine flow demo!")

import jax
jax.config.update('jax_enable_x64', False)

In [ ]:
import jax

# Test if JAX is connected to a GPU
if jax.default_backend() == 'gpu':
    print("✅ JAX is connected to a GPU!")
else:
    print("⚠️ JAX is NOT connected to a GPU. Current backend:", jax.default_backend())


In [ ]:
# Compare TBNN against Carreau-Yasuda (CY as ground truth)
comparison = run_demo_comparison(
    ground_truth_model='carreau_yasuda',
    ground_truth_params=(0.02, 1.0, 5.0, 0.7, 2.0),
    comparison_model='tbnn',
    comparison_params=('tbnn_debug_results_constriction_new/iteration_12_20251008_050525/trajectory_data/final_tbnn_params.pkl',  42),
    domain_size=(256, 256),  # Controls simulation resolution
    dt=5e-5,
    inner_steps=400,
    outer_steps=500,
    pressure_gradient=7.5,
    num_bins=12,
    save_trajectory=False
)

In [ ]:

file_path = str(FROZEN_INST / 'tbnn_debug_results_constriction_new' / 'iteration_12_20251008_050525') + '/'


# Or with error maps (3x2 grid)
result = plot_individual_flow_fields(file_path)


In [ ]:
# Import the script
import parse_job_folders as pjf

# Plot a specific condition (no noise, window 32×16, frozen centers, replicate 1)
result = pjf.plot_piv_field((False, (64, 32), 0.0, 1))

# With noise (4% noise level)
result = pjf.plot_piv_field((False, (32, 16), 0.5, 1))

# With noise (4% noise level)
result = pjf.plot_piv_field((False, (32, 16), 4.0, 1))

# Save to file


In [ ]:

import parse_job_folders as pjf

# Create both plots (display only)
result = pjf.plot_rmse_analysis()



In [ ]:
# Parity plots for n and k (publication quality format)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

# Set publication-quality defaults with larger fonts
# Set publication-quality defaults with larger fonts
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
rcParams['font.size'] = 32
rcParams['axes.labelsize'] = 48
rcParams['axes.titlesize'] = 44
rcParams['xtick.labelsize'] = 40
rcParams['ytick.labelsize'] = 40
rcParams['legend.fontsize'] = 36
rcParams['figure.titlesize'] = 46

# ---- Hard-coded data from your 8 runs ----
runs = ["R1","R2","R3","R4","R5","R6","R7","R8"]

# n
gt_n       = np.array([0.7, 0.6, 0.6, 0.6, 0.8, 0.8, 0.8], dtype=float)
learned_n  = np.array([0.700706, 0.615360,  0.639486, 0.601719,
                       0.791538, 0.793366, 0.795270], dtype=float)

# λ (a.k.a. k/time constant)
gt_lambda      = np.array([5.0, 3.0,  1.0, 7.0, 5.0, 10.0, 15.0], dtype=float)
learned_lambda = np.array([4.941240, 3.030585, 0.982524, 6.626032,
                           4.820224, 9.623342, 14.403651], dtype=float)

# ---- Plot 1: n (linear-linear) ----
fig1, ax1 = plt.subplots(figsize=(10, 10), dpi=600)
ax1.scatter(gt_n, learned_n, s=150, color='#1f77b4', edgecolors='black', linewidth=2, zorder=3)

lo = min(gt_n.min(), learned_n.min())
hi = max(gt_n.max(), learned_n.max())
margin = 0.05 * (hi - lo)
lo -= margin
hi += margin

ax1.plot([lo, hi], [lo, hi], 'k--', linewidth=2.5, zorder=2)
ax1.set_xlim(lo, hi)
ax1.set_ylim(lo, hi)
ax1.set_xlabel('Ground-truth $n$', fontsize=36)
ax1.set_ylabel('Learned $n$', fontsize=36)
ax1.set_aspect('equal', adjustable='box')

# Configure ticks
ax1.grid(False)
ax1.tick_params(axis='both', which='major', labelsize=30, width=2, length=10,
                direction='in', top=True, right=True)
ax1.tick_params(axis='both', which='minor', labelsize=24, width=1.5, length=6,
                direction='in', top=True, right=True)

plt.tight_layout()

# ---- Plot 2: k (log-log) ----
fig2, ax2 = plt.subplots(figsize=(10, 10), dpi=600)
ax2.scatter(gt_lambda, learned_lambda, s=150, color='#ff7f0e', edgecolors='black', linewidth=2, zorder=3)

lo = min(gt_lambda.min(), learned_lambda.min()) * 0.8
hi = max(gt_lambda.max(), learned_lambda.max()) * 1.25
xx = np.logspace(np.log10(lo), np.log10(hi), 256)

ax2.plot(xx, xx, 'k--', linewidth=2.5, zorder=2)
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlim(lo, hi)
ax2.set_ylim(lo, hi)
ax2.set_xlabel('Ground-truth $k$', fontsize=36)
ax2.set_ylabel('Learned $k$', fontsize=36)
ax2.set_aspect('equal', adjustable='box')

# Configure ticks
ax2.grid(False)
ax2.tick_params(axis='both', which='major', labelsize=30, width=2, length=10,
                direction='in', top=True, right=True)
ax2.tick_params(axis='both', which='minor', labelsize=24, width=1.5, length=6,
                direction='in', top=True, right=True)

plt.tight_layout()

# (optional) save copies
# fig1.savefig("parity_n.png", dpi=600, bbox_inches='tight', facecolor='white')
# fig2.savefig("parity_k.png", dpi=600, bbox_inches='tight', facecolor='white')

In [ ]:

import glob

# Find the folder (will error if multiple matches or no matches)
file_path = glob.glob('tbnn_debug_results_constriction_new/noise_iteration_16_*')[0]
print(f"Using: {file_path}")

# Or with error maps (3x2 grid)
result = plot_individual_flow_fields(file_path, noise_used=True)
